<a href="https://colab.research.google.com/github/jahnavichekka/Robot-Arm-Position-Calculator-/blob/main/Robot_Arm_Position_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install flask numpy plotly
print("✅ Packages installed successfully!")

In [2]:

import os
import textwrap

APP_DIR = "/content/robot_arm_pro"
TEMPLATE_DIR = os.path.join(APP_DIR, "templates")

os.makedirs(TEMPLATE_DIR, exist_ok=True)

# ============================================================
# FLASK BACKEND
# ============================================================

app_code = r'''
from flask import Flask, render_template, request, jsonify
import numpy as np
import math

app = Flask(__name__)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def clean_number(value, digits=6):
    value = float(value)

    if abs(value) < 1e-10:
        value = 0.0

    return round(value, digits)


def matrix_to_list(matrix):
    return [
        [clean_number(x) for x in row]
        for row in matrix.tolist()
    ]


def validate_input(data):

    try:
        L1 = float(data.get("L1"))
        L2 = float(data.get("L2"))
        theta1 = float(data.get("theta1"))
        theta2 = float(data.get("theta2"))

        if L1 <= 0:
            return False, "Link 1 length must be greater than 0."

        if L2 <= 0:
            return False, "Link 2 length must be greater than 0."

        if abs(theta1) > 360:
            return False, "Theta 1 must be between -360° and 360°."

        if abs(theta2) > 360:
            return False, "Theta 2 must be between -360° and 360°."

        local_x = data.get("local_x", "")
        local_y = data.get("local_y", "")

        if local_x == "":
            local_x = None
        else:
            local_x = float(local_x)

        if local_y == "":
            local_y = None
        else:
            local_y = float(local_y)

        if (local_x is None) != (local_y is None):
            return False, "Enter both local X and local Y."

        return True, {
            "L1": L1,
            "L2": L2,
            "theta1": theta1,
            "theta2": theta2,
            "local_x": local_x,
            "local_y": local_y
        }

    except (ValueError, TypeError):
        return False, "Please enter valid numerical values."


# ------------------------------------------------------------
# Rotation matrix
# ------------------------------------------------------------

def rotation_matrix(angle_deg):

    angle = math.radians(angle_deg)

    c = math.cos(angle)
    s = math.sin(angle)

    R = np.array([
        [c, -s],
        [s,  c]
    ])

    return R


# ------------------------------------------------------------
# Homogeneous transformation
# ------------------------------------------------------------

def transformation_matrix(length, angle_deg):

    angle = math.radians(angle_deg)

    c = math.cos(angle)
    s = math.sin(angle)

    T = np.array([
        [c, -s, length * c],
        [s,  c, length * s],
        [0,  0,  1]
    ])

    return T


# ------------------------------------------------------------
# Main calculation
# ------------------------------------------------------------

def calculate(data):

    L1 = data["L1"]
    L2 = data["L2"]

    theta1 = data["theta1"]
    theta2 = data["theta2"]

    theta_total = theta1 + theta2

    t1 = math.radians(theta1)
    t2 = math.radians(theta2)
    t_total = math.radians(theta_total)

    # -------------------------
    # Forward kinematics
    # -------------------------

    x1 = L1 * math.cos(t1)
    y1 = L1 * math.sin(t1)

    x2 = (
        L1 * math.cos(t1)
        + L2 * math.cos(t_total)
    )

    y2 = (
        L1 * math.sin(t1)
        + L2 * math.sin(t_total)
    )

    # -------------------------
    # Rotation matrices
    # -------------------------

    R1 = rotation_matrix(theta1)
    R2 = rotation_matrix(theta2)

    R_total = R1 @ R2

    # -------------------------
    # Transformation matrices
    # -------------------------

    T1 = transformation_matrix(L1, theta1)
    T2 = transformation_matrix(L2, theta2)

    T_total = T1 @ T2

    # -------------------------
    # Inverse transformation
    # -------------------------

    try:
        T_inverse = np.linalg.inv(T_total)
        inverse_exists = True
    except np.linalg.LinAlgError:
        T_inverse = None
        inverse_exists = False

    # -------------------------
    # Reach and distance
    # -------------------------

    maximum_reach = L1 + L2

    end_effector_distance = math.sqrt(
        x2 ** 2 + y2 ** 2
    )

    orientation = theta_total

    # -------------------------
    # Local point transformation
    # -------------------------

    local_point = None
    global_point = None

    if (
        data["local_x"] is not None
        and data["local_y"] is not None
    ):

        local_point = np.array([
            data["local_x"],
            data["local_y"],
            1
        ])

        transformed = T_total @ local_point

        global_point = [
            clean_number(transformed[0]),
            clean_number(transformed[1])
        ]

    # -------------------------
    # Calculation steps
    # -------------------------

    steps = [

        {
            "title": "Step 1 — Convert angles to radians",
            "formula": "θ(rad) = θ(deg) × π / 180",
            "calculation":
                f"θ₁ = {theta1} × π / 180 = {clean_number(t1)} rad<br>"
                f"θ₂ = {theta2} × π / 180 = {clean_number(t2)} rad"
        },

        {
            "title": "Step 2 — Calculate Joint 1",
            "formula": "x₁ = L₁ cos(θ₁),  y₁ = L₁ sin(θ₁)",
            "calculation":
                f"x₁ = {L1} × cos({theta1}°) = {clean_number(x1)}<br>"
                f"y₁ = {L1} × sin({theta1}°) = {clean_number(y1)}"
        },

        {
            "title": "Step 3 — Calculate End Effector",
            "formula":
                "x₂ = L₁cos(θ₁) + L₂cos(θ₁+θ₂)<br>"
                "y₂ = L₁sin(θ₁) + L₂sin(θ₁+θ₂)",

            "calculation":
                f"θ₁ + θ₂ = {theta_total}°<br>"
                f"x₂ = {L1}cos({theta1}°) + "
                f"{L2}cos({theta_total}°) = "
                f"{clean_number(x2)}<br>"
                f"y₂ = {L1}sin({theta1}°) + "
                f"{L2}sin({theta_total}°) = "
                f"{clean_number(y2)}"
        },

        {
            "title": "Step 4 — Calculate Reach",
            "formula": "Maximum Reach = L₁ + L₂",
            "calculation":
                f"{L1} + {L2} = "
                f"<strong>{clean_number(maximum_reach)}</strong>"
        },

        {
            "title": "Step 5 — End Effector Distance",
            "formula":
                "d = √(x₂² + y₂²)",

            "calculation":
                f"d = √(({clean_number(x2)})² + "
                f"({clean_number(y2)})²)<br>"
                f"d = <strong>{clean_number(end_effector_distance)}</strong>"
        },

        {
            "title": "Step 6 — End Effector Orientation",
            "formula": "φ = θ₁ + θ₂",
            "calculation":
                f"φ = {theta1}° + {theta2}° = "
                f"<strong>{clean_number(orientation)}°</strong>"
        }
    ]

    return {

        "inputs": {
            "L1": L1,
            "L2": L2,
            "theta1": theta1,
            "theta2": theta2
        },

        "joint1": [
            clean_number(x1),
            clean_number(y1)
        ],

        "end_effector": [
            clean_number(x2),
            clean_number(y2)
        ],

        "maximum_reach": clean_number(maximum_reach),

        "end_effector_distance":
            clean_number(end_effector_distance),

        "orientation":
            clean_number(orientation),

        "R1":
            matrix_to_list(R1),

        "R2":
            matrix_to_list(R2),

        "R_total":
            matrix_to_list(R_total),

        "T1":
            matrix_to_list(T1),

        "T2":
            matrix_to_list(T2),

        "T_total":
            matrix_to_list(T_total),

        "T_inverse":
            matrix_to_list(T_inverse)
            if inverse_exists else None,

        "local_point":
            [
                clean_number(data["local_x"]),
                clean_number(data["local_y"])
            ]
            if data["local_x"] is not None else None,

        "global_point":
            global_point,

        "steps":
            steps
    }


# ------------------------------------------------------------
# Routes
# ------------------------------------------------------------

@app.route("/")
def home():
    return render_template("index.html")


@app.route("/calculate", methods=["POST"])
def calculate_route():

    data = request.get_json()

    valid, result = validate_input(data)

    if not valid:
        return jsonify({
            "success": False,
            "error": result
        }), 400

    try:

        output = calculate(result)

        return jsonify({
            "success": True,
            "result": output
        })

    except Exception as e:

        return jsonify({
            "success": False,
            "error": str(e)
        }), 500


if __name__ == "__main__":

    app.run(
        host="0.0.0.0",
        port=5001,
        debug=False,
        use_reloader=False
    )
'''

with open(os.path.join(APP_DIR, "app.py"), "w", encoding="utf-8") as f:
    f.write(app_code)


# ============================================================
# HTML + CSS + JAVASCRIPT
# ============================================================

html_code = r'''
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>Robot Arm Position Calculator</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

/* =========================================================
   GLOBAL
   ========================================================= */

* {
    box-sizing: border-box;
}

:root {
    --bg: #07111f;
    --bg2: #0b1728;
    --card: rgba(15, 30, 50, 0.88);
    --card2: rgba(19, 39, 63, 0.92);

    --text: #eef6ff;
    --muted: #91a6bc;

    --primary: #4da3ff;
    --primary2: #7c5cff;

    --cyan: #35d6ff;
    --green: #39e58c;
    --orange: #ffb454;
    --red: #ff647c;

    --border: rgba(126, 166, 205, 0.18);

    --shadow:
        0 20px 60px rgba(0, 0, 0, 0.35);
}

body {

    margin: 0;

    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif;

    color: var(--text);

    background:
        radial-gradient(
            circle at 10% 0%,
            rgba(77, 163, 255, 0.15),
            transparent 30%
        ),

        radial-gradient(
            circle at 90% 10%,
            rgba(124, 92, 255, 0.14),
            transparent 28%
        ),

        linear-gradient(
            135deg,
            #050c16,
            #07111f 45%,
            #091625
        );

    min-height: 100vh;
}


/* =========================================================
   NAVBAR
   ========================================================= */

.navbar {

    height: 74px;

    display: flex;

    align-items: center;

    justify-content: space-between;

    padding:
        0 32px;

    border-bottom:
        1px solid var(--border);

    background:
        rgba(5, 13, 24, 0.72);

    backdrop-filter:
        blur(20px);

    position: sticky;

    top: 0;

    z-index: 50;
}

.brand {

    display: flex;

    align-items: center;

    gap: 13px;
}

.brand-icon {

    width: 42px;
    height: 42px;

    display: grid;

    place-items: center;

    border-radius: 13px;

    background:
        linear-gradient(
            135deg,
            var(--primary),
            var(--primary2)
        );

    box-shadow:
        0 10px 30px
        rgba(77, 163, 255, 0.25);

    font-size: 21px;
}

.brand-title {

    font-weight: 800;

    letter-spacing: -0.3px;

    font-size: 17px;
}

.brand-sub {

    font-size: 11px;

    color: var(--muted);

    margin-top: 2px;
}

.system-status {

    display: flex;

    align-items: center;

    gap: 9px;

    color: #b7c8d9;

    font-size: 12px;
}

.status-dot {

    width: 9px;
    height: 9px;

    border-radius: 50%;

    background: var(--green);

    box-shadow:
        0 0 14px
        rgba(57, 229, 140, 0.8);
}


/* =========================================================
   MAIN
   ========================================================= */

.container {

    width: min(1500px, 94%);

    margin: 0 auto;

    padding:
        34px 0 70px;
}


/* =========================================================
   HERO
   ========================================================= */

.hero {

    padding:
        28px 4px 30px;
}

.eyebrow {

    display: inline-flex;

    align-items: center;

    gap: 8px;

    padding:
        7px 12px;

    border:
        1px solid rgba(77, 163, 255, 0.22);

    border-radius: 999px;

    background:
        rgba(77, 163, 255, 0.08);

    color:
        #9bcaff;

    font-size: 11px;

    font-weight: 700;

    text-transform: uppercase;

    letter-spacing: 1.2px;
}

.hero h1 {

    font-size:
        clamp(34px, 5vw, 62px);

    line-height: 1.02;

    letter-spacing: -2.5px;

    margin:
        18px 0 12px;

    max-width: 900px;
}

.gradient-text {

    background:
        linear-gradient(
            90deg,
            #ffffff,
            #74c4ff 45%,
            #a28bff
        );

    -webkit-background-clip: text;

    background-clip: text;

    color: transparent;
}

.hero p {

    color: var(--muted);

    max-width: 850px;

    font-size: 15px;

    line-height: 1.7;

    margin: 0;
}


/* =========================================================
   MAIN GRID
   ========================================================= */

.main-grid {

    display: grid;

    grid-template-columns:
        360px
        minmax(0, 1fr);

    gap: 22px;

    align-items: start;
}


/* =========================================================
   CARDS
   ========================================================= */

.card {

    border:
        1px solid var(--border);

    border-radius: 22px;

    background:
        linear-gradient(
            145deg,
            rgba(17, 35, 57, 0.94),
            rgba(9, 22, 38, 0.94)
        );

    box-shadow: var(--shadow);

    overflow: hidden;
}

.card-header {

    padding:
        20px 22px;

    border-bottom:
        1px solid var(--border);

    display: flex;

    justify-content: space-between;

    align-items: center;
}

.card-header h2 {

    margin: 0;

    font-size: 16px;

    letter-spacing: -0.2px;
}

.card-header span {

    color: var(--muted);

    font-size: 11px;
}

.card-body {

    padding: 22px;
}


/* =========================================================
   CONTROL PANEL
   ========================================================= */

.controls {

    position: sticky;

    top: 94px;
}

.input-group {

    margin-bottom: 18px;
}

.input-group label {

    display: flex;

    justify-content: space-between;

    margin-bottom: 8px;

    color: #cbd9e8;

    font-size: 12px;

    font-weight: 650;
}

.unit {

    color: #70879d;

    font-weight: 500;
}

input {

    width: 100%;

    height: 46px;

    border-radius: 12px;

    border:
        1px solid
        rgba(126, 166, 205, 0.2);

    background:
        rgba(3, 12, 22, 0.65);

    color: white;

    padding:
        0 14px;

    outline: none;

    font-size: 14px;

    transition: 0.2s;
}

input:focus {

    border-color:
        rgba(77, 163, 255, 0.75);

    box-shadow:
        0 0 0 3px
        rgba(77, 163, 255, 0.09);
}

.section-label {

    color: #6f8ba4;

    font-size: 10px;

    font-weight: 800;

    text-transform: uppercase;

    letter-spacing: 1.2px;

    margin:
        24px 0 14px;
}


/* =========================================================
   BUTTONS
   ========================================================= */

.primary-btn {

    width: 100%;

    height: 48px;

    border: 0;

    border-radius: 13px;

    color: white;

    font-weight: 800;

    cursor: pointer;

    background:
        linear-gradient(
            100deg,
            #348ff5,
            #7256ef
        );

    box-shadow:
        0 12px 30px
        rgba(77, 109, 255, 0.22);

    transition:
        transform 0.18s,
        box-shadow 0.18s;
}

.primary-btn:hover {

    transform:
        translateY(-2px);

    box-shadow:
        0 17px 35px
        rgba(77, 109, 255, 0.32);
}

.secondary-btn {

    width: 100%;

    height: 44px;

    border:
        1px solid
        rgba(126, 166, 205, 0.2);

    border-radius: 12px;

    background:
        rgba(255,255,255,0.035);

    color: #b8c8d8;

    font-weight: 700;

    cursor: pointer;

    margin-top: 10px;
}

.secondary-btn:hover {

    background:
        rgba(255,255,255,0.07);

    color: white;
}


/* =========================================================
   EXAMPLES
   ========================================================= */

.examples {

    display: grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap: 7px;
}

.example {

    border:
        1px solid
        rgba(126, 166, 205, 0.16);

    background:
        rgba(255,255,255,0.025);

    border-radius: 10px;

    padding:
        9px 4px;

    color: #9fb1c4;

    cursor: pointer;

    font-size: 10px;

    text-align: center;
}

.example:hover {

    border-color:
        rgba(77, 163, 255, 0.5);

    color: white;
}


/* =========================================================
   VISUALIZATION
   ========================================================= */

.visual-card {

    min-width: 0;
}

.plot-container {

    height:
        560px;

    width: 100%;
}


/* =========================================================
   KPI CARDS
   ========================================================= */

.kpis {

    display: grid;

    grid-template-columns:
        repeat(4, 1fr);

    gap: 14px;

    margin-top: 20px;
}

.kpi {

    padding: 18px;

    border:
        1px solid var(--border);

    border-radius: 17px;

    background:
        rgba(12, 28, 47, 0.78);
}

.kpi-label {

    color: #7790a7;

    font-size: 10px;

    font-weight: 800;

    text-transform: uppercase;

    letter-spacing: 1px;
}

.kpi-value {

    margin-top: 8px;

    font-size: 22px;

    font-weight: 850;

    letter-spacing: -0.7px;
}

.kpi-sub {

    margin-top: 5px;

    color: #748ba1;

    font-size: 10px;
}


/* =========================================================
   TABS
   ========================================================= */

.tabs {

    display: flex;

    gap: 7px;

    overflow-x: auto;

    margin-top: 22px;

    padding-bottom: 2px;
}

.tab {

    white-space: nowrap;

    border:
        1px solid var(--border);

    border-radius: 10px;

    background:
        rgba(255,255,255,0.025);

    color: #8196ab;

    padding:
        10px 14px;

    cursor: pointer;

    font-size: 11px;

    font-weight: 750;
}

.tab.active {

    background:
        rgba(77,163,255,0.12);

    color: #a8d3ff;

    border-color:
        rgba(77,163,255,0.3);
}


/* =========================================================
   CONTENT SECTIONS
   ========================================================= */

.content-panel {

    margin-top: 15px;
}

.tab-content {

    display: none;
}

.tab-content.active {

    display: block;
}


/* =========================================================
   MATRICES
   ========================================================= */

.matrix-grid {

    display: grid;

    grid-template-columns:
        repeat(2, 1fr);

    gap: 15px;
}

.matrix-card {

    padding: 17px;

    border:
        1px solid var(--border);

    border-radius: 15px;

    background:
        rgba(3, 12, 22, 0.48);
}

.matrix-title {

    color: #a7bfd6;

    font-size: 11px;

    font-weight: 800;

    margin-bottom: 13px;
}

.matrix {

    display: grid;

    gap: 5px;
}

.matrix-row {

    display: grid;

    gap: 5px;
}

.matrix-cell {

    padding:
        9px 6px;

    border-radius: 7px;

    background:
        rgba(255,255,255,0.035);

    text-align: center;

    font-family:
        "JetBrains Mono",
        "Consolas",
        monospace;

    font-size: 10px;

    color: #d9e8f7;
}


/* =========================================================
   SOLUTION
   ========================================================= */

.step {

    padding: 20px;

    border:
        1px solid var(--border);

    border-radius: 15px;

    margin-bottom: 12px;

    background:
        rgba(255,255,255,0.025);
}

.step-number {

    color: #6caeff;

    font-size: 10px;

    font-weight: 850;

    text-transform: uppercase;

    letter-spacing: 1px;
}

.step h3 {

    margin:
        7px 0 10px;

    font-size: 14px;
}

.formula {

    display: inline-block;

    padding:
        10px 13px;

    border-radius: 9px;

    background:
        rgba(77,163,255,0.08);

    color: #b9dcff;

    font-family:
        "Consolas",
        monospace;

    font-size: 11px;

    line-height: 1.7;
}

.calculation {

    margin-top: 12px;

    color: #8fa4b9;

    font-size: 12px;

    line-height: 1.8;
}


/* =========================================================
   RESULT BANNER
   ========================================================= */

.result-banner {

    margin-bottom: 18px;

    padding: 22px;

    border:
        1px solid
        rgba(57,229,140,0.18);

    border-radius: 18px;

    background:
        linear-gradient(
            110deg,
            rgba(57,229,140,0.07),
            rgba(77,163,255,0.06)
        );
}

.result-banner h3 {

    margin: 0 0 7px;

    font-size: 17px;
}

.result-banner p {

    margin: 0;

    color: #88a0b5;

    font-size: 12px;
}


/* =========================================================
   ERROR
   ========================================================= */

.error {

    display: none;

    padding: 13px 15px;

    border-radius: 11px;

    margin-bottom: 15px;

    background:
        rgba(255,100,124,0.09);

    border:
        1px solid
        rgba(255,100,124,0.25);

    color: #ff9eac;

    font-size: 12px;
}


/* =========================================================
   ABOUT
   ========================================================= */

.about-grid {

    display: grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap: 14px;
}

.about-item {

    padding: 18px;

    border:
        1px solid var(--border);

    border-radius: 15px;

    background:
        rgba(255,255,255,0.025);
}

.about-item h4 {

    margin:
        0 0 8px;

    font-size: 12px;
}

.about-item p {

    margin: 0;

    color: #8499ae;

    font-size: 11px;

    line-height: 1.7;
}


/* =========================================================
   FOOTER
   ========================================================= */

.footer {

    text-align: center;

    color: #5e758c;

    font-size: 10px;

    margin-top: 38px;
}


/* =========================================================
   RESPONSIVE
   ========================================================= */

@media(max-width: 1100px) {

    .main-grid {

        grid-template-columns: 1fr;
    }

    .controls {

        position: relative;

        top: auto;
    }

    .plot-container {

        height: 500px;
    }
}

@media(max-width: 750px) {

    .navbar {

        padding: 0 17px;
    }

    .container {

        width: 94%;
    }

    .kpis {

        grid-template-columns:
            repeat(2, 1fr);
    }

    .matrix-grid {

        grid-template-columns: 1fr;
    }

    .about-grid {

        grid-template-columns: 1fr;
    }

    .plot-container {

        height: 430px;
    }

    .hero h1 {

        letter-spacing: -1.5px;
    }
}

</style>

</head>


<body>


<!-- ======================================================
     NAVBAR
     ====================================================== -->

<nav class="navbar">

    <div class="brand">

        <div class="brand-icon">
            ⚙
        </div>

        <div>

            <div class="brand-title">
                Robot Arm Calculator
            </div>

            <div class="brand-sub">
                Coordinate Transformation Engine
            </div>

        </div>

    </div>


    <div class="system-status">

        <span class="status-dot"></span>

        SYSTEM ONLINE

    </div>

</nav>


<!-- ======================================================
     MAIN
     ====================================================== -->

<main class="container">


    <!-- HERO -->

    <section class="hero">

        <div class="eyebrow">
            F2 · ROBOTICS · LINEAR ALGEBRA
        </div>

        <h1>
            <span class="gradient-text">
                Robot Arm Position
            </span>
            Calculator
        </h1>

        <p>
            Analyze a two-link planar robotic arm using
            forward kinematics, rotation matrices,
            homogeneous transformation matrices and
            coordinate transformations.
        </p>

    </section>


    <!-- MAIN GRID -->

    <section class="main-grid">


        <!-- ==================================================
             LEFT CONTROL PANEL
             ================================================== -->

        <aside class="controls">

            <div class="card">

                <div class="card-header">

                    <h2>
                        Arm Configuration
                    </h2>

                    <span>
                        INPUT
                    </span>

                </div>


                <div class="card-body">

                    <div id="errorBox"
                         class="error">
                    </div>


                    <div class="section-label">
                        Link Parameters
                    </div>


                    <div class="input-group">

                        <label>
                            Link 1 Length
                            <span class="unit">
                                units
                            </span>
                        </label>

                        <input
                            id="L1"
                            type="number"
                            step="any"
                            value="5"
                        >

                    </div>


                    <div class="input-group">

                        <label>
                            Link 2 Length
                            <span class="unit">
                                units
                            </span>
                        </label>

                        <input
                            id="L2"
                            type="number"
                            step="any"
                            value="3"
                        >

                    </div>


                    <div class="section-label">
                        Joint Angles
                    </div>


                    <div class="input-group">

                        <label>
                            θ₁ — Base Joint
                            <span class="unit">
                                degrees
                            </span>
                        </label>

                        <input
                            id="theta1"
                            type="number"
                            step="any"
                            value="0"
                        >

                    </div>


                    <div class="input-group">

                        <label>
                            θ₂ — Elbow Joint
                            <span class="unit">
                                degrees
                            </span>
                        </label>

                        <input
                            id="theta2"
                            type="number"
                            step="any"
                            value="0"
                        >

                    </div>


                    <div class="section-label">
                        Optional Local Point
                    </div>


                    <div class="input-group">

                        <label>
                            Local X
                            <span class="unit">
                                optional
                            </span>
                        </label>

                        <input
                            id="localX"
                            type="number"
                            step="any"
                            placeholder="e.g. 1"
                        >

                    </div>


                    <div class="input-group">

                        <label>
                            Local Y
                            <span class="unit">
                                optional
                            </span>
                        </label>

                        <input
                            id="localY"
                            type="number"
                            step="any"
                            placeholder="e.g. 2"
                        >

                    </div>


                    <button
                        class="primary-btn"
                        onclick="calculateArm()">

                        Calculate Position

                    </button>


                    <button
                        class="secondary-btn"
                        onclick="resetForm()">

                        Reset

                    </button>


                    <div class="section-label">
                        Quick Examples
                    </div>


                    <div class="examples">

                        <button
                            class="example"
                            onclick="example1()">
                            Straight
                        </button>

                        <button
                            class="example"
                            onclick="example2()">
                            Vertical
                        </button>

                        <button
                            class="example"
                            onclick="example3()">
                            90° Elbow
                        </button>

                    </div>

                </div>

            </div>

        </aside>


        <!-- ==================================================
             RIGHT SIDE
             ================================================== -->

        <section>


            <!-- VISUALIZATION -->

            <div class="card visual-card">

                <div class="card-header">

                    <h2>
                        Live Robot Visualization
                    </h2>

                    <span>
                        FORWARD KINEMATICS
                    </span>

                </div>

                <div
                    id="plot"
                    class="plot-container">
                </div>

            </div>


            <!-- KPI -->

            <div class="kpis">


                <div class="kpi">

                    <div class="kpi-label">
                        Joint 1
                    </div>

                    <div
                        id="jointValue"
                        class="kpi-value">
                        (0, 0)
                    </div>

                    <div class="kpi-sub">
                        J1 coordinates
                    </div>

                </div>


                <div class="kpi">

                    <div class="kpi-label">
                        End Effector
                    </div>

                    <div
                        id="eeValue"
                        class="kpi-value">
                        (0, 0)
                    </div>

                    <div class="kpi-sub">
                        final position
                    </div>

                </div>


                <div class="kpi">

                    <div class="kpi-label">
                        Distance
                    </div>

                    <div
                        id="distanceValue"
                        class="kpi-value">
                        0
                    </div>

                    <div class="kpi-sub">
                        from origin
                    </div>

                </div>


                <div class="kpi">

                    <div class="kpi-label">
                        Orientation
                    </div>

                    <div
                        id="orientationValue"
                        class="kpi-value">
                        0°
                    </div>

                    <div class="kpi-sub">
                        θ₁ + θ₂
                    </div>

                </div>

            </div>


            <!-- TABS -->

            <div class="tabs">

                <button
                    class="tab active"
                    onclick="openTab(event, 'overview')">

                    Overview

                </button>

                <button
                    class="tab"
                    onclick="openTab(event, 'matrices')">

                    Matrices

                </button>

                <button
                    class="tab"
                    onclick="openTab(event, 'solution')">

                    Step-by-Step

                </button>

                <button
                    class="tab"
                    onclick="openTab(event, 'transform')">

                    Coordinate Transform

                </button>

                <button
                    class="tab"
                    onclick="openTab(event, 'about')">

                    About

                </button>

            </div>


            <!-- TAB CONTENT -->

            <div class="content-panel">


                <!-- OVERVIEW -->

                <div
                    id="overview"
                    class="tab-content active">

                    <div class="result-banner">

                        <h3>
                            Calculation Complete
                        </h3>

                        <p>
                            The robot arm position has been
                            calculated using forward kinematics.
                            The diagram above represents the
                            calculated joint coordinates.
                        </p>

                    </div>


                    <div class="matrix-grid">


                        <div class="matrix-card">

                            <div class="matrix-title">
                                Forward Kinematics
                            </div>

                            <div class="formula">

                                x₁ = L₁ cos θ₁<br>

                                y₁ = L₁ sin θ₁<br><br>

                                x₂ =
                                L₁cosθ₁ +
                                L₂cos(θ₁+θ₂)<br>

                                y₂ =
                                L₁sinθ₁ +
                                L₂sin(θ₁+θ₂)

                            </div>

                        </div>


                        <div class="matrix-card">

                            <div class="matrix-title">
                                Robot Geometry
                            </div>

                            <div class="calculation">

                                <strong>
                                    Maximum Reach:
                                </strong>

                                <span id="reachValue">
                                    0
                                </span>

                                <br><br>

                                <strong>
                                    Link 1:
                                </strong>

                                <span id="link1Value">
                                    0
                                </span>

                                <br><br>

                                <strong>
                                    Link 2:
                                </strong>

                                <span id="link2Value">
                                    0
                                </span>

                            </div>

                        </div>

                    </div>

                </div>


                <!-- MATRICES -->

                <div
                    id="matrices"
                    class="tab-content">

                    <div
                        id="matrixContainer"
                        class="matrix-grid">
                    </div>

                </div>


                <!-- SOLUTION -->

                <div
                    id="solution"
                    class="tab-content">

                    <div
                        id="stepsContainer">
                    </div>

                </div>


                <!-- TRANSFORM -->

                <div
                    id="transform"
                    class="tab-content">

                    <div class="result-banner">

                        <h3>
                            Local → Global Coordinate Transformation
                        </h3>

                        <p>
                            A local homogeneous coordinate can
                            be transformed into the global
                            coordinate system using:
                            Pglobal = T × Plocal
                        </p>

                    </div>


                    <div
                        id="transformResult"
                        class="matrix-card">

                        Enter Local X and Local Y in the
                        control panel to calculate the
                        transformed global point.

                    </div>

                </div>


                <!-- ABOUT -->

                <div
                    id="about"
                    class="tab-content">

                    <div class="about-grid">


                        <div class="about-item">

                            <h4>
                                Matrix Multiplication
                            </h4>

                            <p>
                                Transformation matrices are
                                multiplied to combine the
                                movements and rotations of
                                the two robot links.
                            </p>

                        </div>


                        <div class="about-item">

                            <h4>
                                Rotation Matrices
                            </h4>

                            <p>
                                Rotation matrices represent
                                the orientation of each robot
                                joint in a 2D coordinate system.
                            </p>

                        </div>


                        <div class="about-item">

                            <h4>
                                Homogeneous Coordinates
                            </h4>

                            <p>
                                A 3×3 homogeneous matrix
                                combines rotation and
                                translation into one operation.
                            </p>

                        </div>


                        <div class="about-item">

                            <h4>
                                Technology
                            </h4>

                            <p>
                                Python, Flask, NumPy,
                                HTML, CSS, JavaScript
                                and Plotly.
                            </p>

                        </div>


                        <div class="about-item">

                            <h4>
                                Application
                            </h4>

                            <p>
                                Useful for understanding
                                robotics, coordinate systems,
                                linear algebra and forward
                                kinematics.
                            </p>

                        </div>


                        <div class="about-item">

                            <h4>
                                Interactive Analysis
                            </h4>

                            <p>
                                Change the link lengths or
                                joint angles and immediately
                                visualize the new robot position.
                            </p>

                        </div>

                    </div>

                </div>


            </div>

        </section>

    </section>


    <div class="footer">

        Robot Arm Position Calculator ·
        2-Link Planar Robotics Analysis

    </div>

</main>


<script>

/* =========================================================
   GLOBAL
   ========================================================= */

let currentResult = null;


/* =========================================================
   NUMBER FORMAT
   ========================================================= */

function fmt(value) {

    if (Math.abs(value) < 0.000001) {
        value = 0;
    }

    return Number(value).toFixed(4);
}


/* =========================================================
   CALCULATE
   ========================================================= */

async function calculateArm() {

    const errorBox =
        document.getElementById("errorBox");

    errorBox.style.display = "none";


    const L1 =
        document.getElementById("L1").value;

    const L2 =
        document.getElementById("L2").value;

    const theta1 =
        document.getElementById("theta1").value;

    const theta2 =
        document.getElementById("theta2").value;

    const localX =
        document.getElementById("localX").value;

    const localY =
        document.getElementById("localY").value;


    try {

        const response =
            await fetch("/calculate", {

                method: "POST",

                headers: {
                    "Content-Type":
                        "application/json"
                },

                body: JSON.stringify({

                    L1: L1,
                    L2: L2,

                    theta1: theta1,
                    theta2: theta2,

                    local_x: localX,
                    local_y: localY

                })

            });


        const data =
            await response.json();


        if (!response.ok ||
            !data.success) {

            throw new Error(
                data.error ||
                "Calculation failed."
            );

        }


        currentResult =
            data.result;


        updateInterface(
            currentResult
        );


    } catch (error) {

        errorBox.innerHTML =
            "⚠ " + error.message;

        errorBox.style.display =
            "block";

    }

}


/* =========================================================
   UPDATE INTERFACE
   ========================================================= */

function updateInterface(result) {

    const j1 =
        result.joint1;

    const ee =
        result.end_effector;


    document.getElementById(
        "jointValue"
    ).textContent =
        `(${fmt(j1[0])}, ${fmt(j1[1])})`;


    document.getElementById(
        "eeValue"
    ).textContent =
        `(${fmt(ee[0])}, ${fmt(ee[1])})`;


    document.getElementById(
        "distanceValue"
    ).textContent =
        fmt(result.end_effector_distance);


    document.getElementById(
        "orientationValue"
    ).textContent =
        `${fmt(result.orientation)}°`;


    document.getElementById(
        "reachValue"
    ).textContent =
        fmt(result.maximum_reach);


    document.getElementById(
        "link1Value"
    ).textContent =
        result.inputs.L1;


    document.getElementById(
        "link2Value"
    ).textContent =
        result.inputs.L2;


    drawRobot(result);

    renderMatrices(result);

    renderSteps(result);

    renderTransform(result);

}


/* =========================================================
   DRAW ROBOT
   ========================================================= */

function drawRobot(result) {

    const L1 =
        result.inputs.L1;

    const L2 =
        result.inputs.L2;

    const j1 =
        result.joint1;

    const ee =
        result.end_effector;


    const reach =
        L1 + L2;


    const margin =
        Math.max(
            1,
            reach * 0.25
        );


    const axis =
        reach + margin;


    const robotTrace = {

        x: [
            0,
            j1[0],
            ee[0]
        ],

        y: [
            0,
            j1[1],
            ee[1]
        ],

        mode: "lines+markers",

        line: {
            width: 12
        },

        marker: {

            size: [
                17,
                20,
                24
            ]

        },

        hovertemplate:
            "<b>Robot Point</b><br>" +
            "X: %{x:.3f}<br>" +
            "Y: %{y:.3f}" +
            "<extra></extra>"
    };


    const baseTrace = {

        x: [0],

        y: [0],

        mode: "markers",

        marker: {

            size: 30,

            symbol: "circle"

        },

        name: "Base",

        hovertemplate:
            "<b>Base</b><extra></extra>"
    };


    const jointTrace = {

        x: [j1[0]],

        y: [j1[1]],

        mode: "markers",

        marker: {

            size: 18

        },

        name: "Joint 1",

        hovertemplate:
            "<b>Joint 1</b><br>" +
            "X: %{x:.3f}<br>" +
            "Y: %{y:.3f}" +
            "<extra></extra>"
    };


    const eeTrace = {

        x: [ee[0]],

        y: [ee[1]],

        mode: "markers+text",

        text: ["END EFFECTOR"],

        textposition: "top center",

        marker: {

            size: 19,

            symbol: "diamond"

        },

        name: "End Effector",

        hovertemplate:
            "<b>End Effector</b><br>" +
            "X: %{x:.3f}<br>" +
            "Y: %{y:.3f}" +
            "<extra></extra>"
    };


    const xAxisTrace = {

        x: [-axis, axis],

        y: [0, 0],

        mode: "lines",

        line: {
            width: 1
        },

        name: "X axis",

        hoverinfo: "skip"
    };


    const yAxisTrace = {

        x: [0, 0],

        y: [-axis, axis],

        mode: "lines",

        line: {
            width: 1
        },

        name: "Y axis",

        hoverinfo: "skip"
    };


    const data = [

        xAxisTrace,

        yAxisTrace,

        robotTrace,

        baseTrace,

        jointTrace,

        eeTrace

    ];


    const layout = {

        margin: {
            l: 30,
            r: 30,
            t: 25,
            b: 30
        },

        paper_bgcolor:
            "rgba(0,0,0,0)",

        plot_bgcolor:
            "rgba(0,0,0,0)",

        font: {
            color: "#a9bdd1"
        },

        showlegend: true,

        legend: {
            orientation: "h",
            y: -0.05
        },

        xaxis: {

            title: "X Coordinate",

            range: [
                -axis,
                axis
            ],

            zeroline: false,

            gridcolor:
                "rgba(130,160,190,0.12)",

            color: "#71869b",

            scaleanchor: "y",

            scaleratio: 1
        },

        yaxis: {

            title: "Y Coordinate",

            range: [
                -axis,
                axis
            ],

            zeroline: false,

            gridcolor:
                "rgba(130,160,190,0.12)",

            color: "#71869b"
        },

        hovermode: "closest",

        dragmode: "pan"
    };


    Plotly.react(
        "plot",
        data,
        layout,
        {
            responsive: true,
            displaylogo: false,
            modeBarButtonsToRemove: [
                "lasso2d",
                "select2d"
            ]
        }
    );

}


/* =========================================================
   MATRIX RENDER
   ========================================================= */

function renderMatrix(
    title,
    matrix
) {

    let html = `
        <div class="matrix-card">
            <div class="matrix-title">
                ${title}
            </div>
            <div class="matrix">
    `;


    matrix.forEach(row => {

        html += `
            <div
                class="matrix-row"
                style="
                    grid-template-columns:
                    repeat(${row.length}, 1fr);
                "
            >
        `;


        row.forEach(value => {

            html += `
                <div class="matrix-cell">
                    ${fmt(value)}
                </div>
            `;

        });


        html += `
            </div>
        `;

    });


    html += `
            </div>
        </div>
    `;


    return html;
}


function renderMatrices(result) {

    const container =
        document.getElementById(
            "matrixContainer"
        );


    container.innerHTML =

        renderMatrix(
            "Rotation Matrix R₁",
            result.R1
        ) +

        renderMatrix(
            "Rotation Matrix R₂",
            result.R2
        ) +

        renderMatrix(
            "Combined Rotation R₁ × R₂",
            result.R_total
        ) +

        renderMatrix(
            "Transformation Matrix T₁",
            result.T1
        ) +

        renderMatrix(
            "Transformation Matrix T₂",
            result.T2
        ) +

        renderMatrix(
            "Final Transformation T₁ × T₂",
            result.T_total
        ) +

        renderMatrix(
            "Inverse Transformation T⁻¹",
            result.T_inverse
        );

}


/* =========================================================
   STEPS
   ========================================================= */

function renderSteps(result) {

    const container =
        document.getElementById(
            "stepsContainer"
        );


    container.innerHTML = "";


    result.steps.forEach(
        (step, index) => {

            const div =
                document.createElement(
                    "div"
                );

            div.className =
                "step";


            div.innerHTML = `

                <div class="step-number">
                    ${index + 1}
                </div>

                <h3>
                    ${step.title}
                </h3>

                <div class="formula">
                    ${step.formula}
                </div>

                <div class="calculation">
                    ${step.calculation}
                </div>

            `;


            container.appendChild(div);

        }
    );

}


/* =========================================================
   TRANSFORMATION
   ========================================================= */

function renderTransform(result) {

    const container =
        document.getElementById(
            "transformResult"
        );


    if (!result.local_point) {

        container.innerHTML = `

            <div style="
                color:#8499ae;
                line-height:1.8;
                font-size:12px;
            ">

                No local point entered.

                <br><br>

                Enter Local X and Local Y
                on the left to calculate:

                <br><br>

                <div class="formula">
                    Pglobal = T × Plocal
                </div>

            </div>

        `;

        return;
    }


    const local =
        result.local_point;

    const global =
        result.global_point;


    container.innerHTML = `

        <div class="matrix-title">
            Transformation Result
        </div>

        <div class="calculation">

            Local Point:

            <strong>
                (${fmt(local[0])},
                 ${fmt(local[1])})
            </strong>

            <br><br>

            Final Transformation:

            <strong>
                T × P
            </strong>

            <br><br>

            Global Point:

            <strong style="
                color:#7fe8ad;
                font-size:16px;
            ">
                (${fmt(global[0])},
                 ${fmt(global[1])})
            </strong>

        </div>

    `;

}


/* =========================================================
   RESET
   ========================================================= */

function resetForm() {

    document.getElementById(
        "L1"
    ).value = 5;

    document.getElementById(
        "L2"
    ).value = 3;

    document.getElementById(
        "theta1"
    ).value = 0;

    document.getElementById(
        "theta2"
    ).value = 0;

    document.getElementById(
        "localX"
    ).value = "";

    document.getElementById(
        "localY"
    ).value = "";


    calculateArm();

}


/* =========================================================
   EXAMPLES
   ========================================================= */

function example1() {

    document.getElementById("L1").value = 5;

    document.getElementById("L2").value = 3;

    document.getElementById("theta1").value = 0;

    document.getElementById("theta2").value = 0;

    document.getElementById("localX").value = "";

    document.getElementById("localY").value = "";

    calculateArm();
}


function example2() {

    document.getElementById("L1").value = 5;

    document.getElementById("L2").value = 3;

    document.getElementById("theta1").value = 90;

    document.getElementById("theta2").value = 0;

    document.getElementById("localX").value = "";

    document.getElementById("localY").value = "";

    calculateArm();
}


function example3() {

    document.getElementById("L1").value = 5;

    document.getElementById("L2").value = 3;

    document.getElementById("theta1").value = 0;

    document.getElementById("theta2").value = 90;

    document.getElementById("localX").value = "";

    document.getElementById("localY").value = "";

    calculateArm();
}


/* =========================================================
   TABS
   ========================================================= */

function openTab(
    event,
    tabId
) {

    document
        .querySelectorAll(
            ".tab-content"
        )
        .forEach(content => {

            content.classList.remove(
                "active"
            );

        });


    document
        .querySelectorAll(
            ".tab"
        )
        .forEach(tab => {

            tab.classList.remove(
                "active"
            );

        });


    document
        .getElementById(tabId)
        .classList.add("active");


    event.currentTarget
        .classList.add("active");

}


/* =========================================================
   AUTO CALCULATE
   ========================================================= */

window.addEventListener(
    "load",
    () => {

        calculateArm();

    }
);

</script>

</body>

</html>
'''

with open(
    os.path.join(TEMPLATE_DIR, "index.html"),
    "w",
    encoding="utf-8"
) as f:

    f.write(html_code)


print("✅ Professional Robot Arm application created!")
print("📁 Location:", APP_DIR)
print("🌐 Port: 5001")

import sys
import threading
import time

sys.path.insert(0, "/content/robot_arm_pro")

from app import app

def run_app():
    app.run(
        host="0.0.0.0",
        port=5001,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(
    target=run_app,
    daemon=True
)

thread.start()

time.sleep(2)

print("🚀 Robot Arm Calculator is running!")
print("🌐 Opening the application...")

from google.colab import output

output.serve_kernel_port_as_iframe(
    5001,
    height=1000
)

✅ Professional Robot Arm application created!
📁 Location: /content/robot_arm_pro
🌐 Port: 5001
 * Serving Flask app 'app'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


🚀 Robot Arm Calculator is running!
🌐 Opening the application...


<IPython.core.display.Javascript object>